In [ ]:
import xarray as xr
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import multiprocessing as mp
import random
from shapely.geometry import box

# Sanitation Check

In [ ]:
#Portugal Shapefile
pt = gpd.read_file("/home/lixinh/Study_Area/Portugal_Boundary/Portugal_island_removed.shp")
pt.boundary.plot(color = "black")

In [ ]:
#FRY 6D
fry_2017_6D = gpd.read_file("/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_6D/FRY_fire_patches_POLYGON_6D_2017.shp")
fry_2017_6D = fry_2017_6D.to_crs(epsg = 4326)
fry_2017_6D_pt = gpd.clip(fry_2017_6D, pt)
fry_2017_6D_pt.plot()

In [ ]:
#FRY 12D
fry_2017_12D = gpd.read_file("/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_12D/FRY_fire_patches_POLYGON_12D_2017.shp")
fry_2017_12D = fry_2017_12D.to_crs(epsg = 4326)
fry_2017_12D_pt = gpd.clip(fry_2017_12D, pt)
fry_2017_12D_pt.plot()

In [ ]:
cmap = plt.get_cmap('tab20')  

# Plot each polygon with a different color
fig, (ax1, ax2) = plt.subplots(1,2,figsize = (26,13))

#6D
pt.boundary.plot(ax = ax1, edgecolor = "black")
for i, polygon in enumerate(fry_2017_6D_pt['geometry']):
    color = cmap(i % cmap.N)  # Cycle through the colormap
    fry_2017_6D_pt.iloc[[i]].plot(ax=ax1, color=color)
ax1.set_ylabel("Lat", fontsize = 12)
ax1.set_xlabel("Lon", fontsize = 12)
ax1.set_title("FRY-6D 2017", fontsize = 15)

#12D
pt.boundary.plot(ax = ax2, edgecolor = "black")
for i, polygon in enumerate(fry_2017_12D_pt['geometry']):
    color = cmap(i % cmap.N)  # Cycle through the colormap
    fry_2017_12D_pt.iloc[[i]].plot(ax=ax2, color=color)
ax2.set_ylabel("Lat", fontsize = 12)
ax2.set_xlabel("Lon", fontsize = 12)
ax2.set_title("FRY-12D 2017", fontsize = 15)

plt.subplots_adjust(wspace=0.0) 
plt.show()

# Stack and clip yearly fire polygons with EU Bounding Box (-11, 33, 35, 72, minlon, minlat, maxlon, maxlat)

## Function

In [ ]:
def read_and_clip(filepath, mask):
    data = gpd.read_file(filepath)
    data = data.to_crs(epsg = 4326)
    data_clipped = gpd.clip(data, mask)
    return data_clipped

## EU Bounding BOX

In [ ]:
bbox = box(-11, 33, 35, 72)

In [ ]:
bbox

## 6D product

In [ ]:
fry_6d_paths = [rf"/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_6D/FRY_fire_patches_POLYGON_6D_{yr}.shp" for yr in range(2001, 2021)]

In [ ]:
fry_6d_paths

In [ ]:
pool = mp.Pool(20)
fry_6d_stack_eubbox = pd.concat([pool.apply(read_and_clip, args=(path, bbox)) for path in fry_6d_paths], ignore_index = True)
pool.close()

In [ ]:
fry_6d_stack_eubbox

In [ ]:
#save
fry_6d_stack_eubbox.to_file("/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_6D/FRYv2.0_FireCCI51_6D_2001-2020_EUbbox.shp")

## 12D product

In [ ]:
fry_12d_paths = [rf"/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_12D/FRY_fire_patches_POLYGON_12D_{yr}.shp" for yr in range(2001, 2021)]

In [ ]:
fry_12d_paths

In [ ]:
pool = mp.Pool(20)
fry_12d_stack_eubbox = pd.concat([pool.apply(read_and_clip, args=(path, bbox)) for path in fry_12d_paths], ignore_index = True)
pool.close()

In [ ]:
fry_12d_stack_eubbox

In [ ]:
#save
fry_12d_stack_eubbox.to_file("/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_12D/FRYv2.0_FireCCI51_12D_2001-2020_EUbbox.shp")

## Check CRS

In [ ]:
shp_6d = gpd.read_file("/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_6D/FRYv2.0_FireCCI51_6D_2001-2020_EUbbox.shp")

In [ ]:
shp_12d = gpd.read_file("/net/krypton/hyclimm/data/public/hazard/obs/fire_observations/FRYv2.0_FireCCI51/SHP_12D/FRYv2.0_FireCCI51_12D_2001-2020_EUbbox.shp")

In [ ]:
shp_6d.crs

In [ ]:
shp_12d.crs